# Nigerian Retail Transaction Fraud Analysis — Data Cleaning

This notebook uses **DuckDB** to explore, validate, and clean a ~5 million row dataset of Nigerian retail transactions (stored as Parquet), then exports the cleaned data as a CSV for visualization in the [Fraud Transaction Intelligence Tableau dashboard](https://public.tableau.com/app/profile/fechi.iroegbu/viz/FinancialFraudAnalysisThreatIntelligenceDashboard/Dashboard1).

**Workflow:**
1. Load and explore the raw data
2. Run data quality checks (duplicates, missing values)
3. Explore key breakdowns (channel, transaction type, location)
4. Clean and standardize the data
5. Export the cleaned dataset for Tableau

## Setup

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install duckdb pandas

import os
import duckdb

# Path to the raw parquet file — update this to point to your local copy
file_path = os.path.join(os.path.expanduser("~"), "Documents", "nigerian_retail_transactions_full.parquet")

## 1. Explore the Raw Data

In [ ]:
# Preview the first few rows
duckdb.query(f"SELECT * FROM '{file_path}' LIMIT 5").df()

In [ ]:
# Row count and schema
row_count = duckdb.query(f"SELECT COUNT(*) FROM '{file_path}'").fetchone()[0]
print(f"Total rows: {row_count:,}")

duckdb.query(f"DESCRIBE SELECT * FROM '{file_path}'").df()

## 2. Data Quality Checks

In [ ]:
# Check for duplicate transaction IDs
duckdb.query(f"""
    SELECT transaction_id, COUNT(*) AS occurrences
    FROM '{file_path}'
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
""").df()

In [ ]:
# Count missing values across key columns
duckdb.query(f"""
    SELECT
        COUNT(*) - COUNT(transaction_id)       AS null_transaction_ids,
        COUNT(*) - COUNT(account_id)           AS null_account_ids,
        COUNT(*) - COUNT(customer_id)          AS null_customer_ids,
        COUNT(*) - COUNT(timestamp)            AS null_timestamps,
        COUNT(*) - COUNT(amount_ngn)           AS null_amounts,
        COUNT(*) - COUNT(transaction_type)     AS null_transaction_types,
        COUNT(*) - COUNT(channel)              AS null_channels,
        COUNT(*) - COUNT(merchant_category_code) AS null_mccs,
        COUNT(*) - COUNT(merchant_name)        AS null_merchants,
        COUNT(*) - COUNT(location_lga)         AS null_locations
    FROM '{file_path}'
""").df()

## 3. Exploratory Breakdown

In [ ]:
# Transaction count by channel
duckdb.query(f"""
    SELECT channel, COUNT(*) AS total_count
    FROM '{file_path}'
    GROUP BY channel
    ORDER BY total_count DESC
""").df()

In [ ]:
# Transaction count by type
duckdb.query(f"""
    SELECT transaction_type, COUNT(*) AS total_count
    FROM '{file_path}'
    GROUP BY transaction_type
    ORDER BY total_count DESC
""").df()

In [ ]:
# Transaction count by location (LGA)
duckdb.query(f"""
    SELECT location_lga, COUNT(*) AS total_count
    FROM '{file_path}'
    GROUP BY location_lga
    ORDER BY total_count DESC
""").df()

## 4. Clean the Data

Based on the checks above, the cleaning steps applied are:
- Cast `amount_ngn`, `balance_before_ngn`, and `balance_after_ngn` to `BIGINT` (removing unnecessary decimal precision)
- Fill blank/missing `merchant_category_code` values with `'N/A'`
- Fill blank/missing `merchant_name` values with `'Non_merchant'`

In [ ]:
# Preview the cleaned data before exporting
duckdb.query(f"""
    SELECT * REPLACE (
        CAST(amount_ngn AS BIGINT) AS amount_ngn,
        CAST(balance_before_ngn AS BIGINT) AS balance_before_ngn,
        CAST(balance_after_ngn AS BIGINT) AS balance_after_ngn,
        COALESCE(NULLIF(TRIM(merchant_category_code), ''), 'N/A') AS merchant_category_code,
        COALESCE(NULLIF(TRIM(merchant_name), ''), 'Non_merchant') AS merchant_name
    )
    FROM '{file_path}'
""").df()

## 5. Export Cleaned Data for Tableau

In [ ]:
output_path = "cleaned_retail_transactions.csv"

duckdb.query(f"""
    COPY (
        SELECT * REPLACE (
            CAST(amount_ngn AS BIGINT) AS amount_ngn,
            CAST(balance_before_ngn AS BIGINT) AS balance_before_ngn,
            CAST(balance_after_ngn AS BIGINT) AS balance_after_ngn,
            COALESCE(NULLIF(TRIM(merchant_category_code), ''), 'N/A') AS merchant_category_code,
            COALESCE(NULLIF(TRIM(merchant_name), ''), 'Non_merchant') AS merchant_name
        )
        FROM '{file_path}'
    ) TO '{output_path}' (HEADER, DELIMITER ',');
""")

print(f"Export complete! '{output_path}' saved and ready to load into Tableau.")